# 讲课笔记：Day 1 上午 - 从文本到向量（3小时）

## 讲师备课指南

本笔记对应 Notebook：`Day1_上午_从文本到向量.ipynb`（共43个Cell，编号0-42）

---

### 整体时间分配

| 时间段 | 内容 | 时长 | 对应Cell |
|--------|------|------|----------|
| 09:00-09:10 | 开场 + 课程总览 + 环境检查 | 10 min | Cell 0-2 |
| 09:10-09:40 | 第一部分：训练循环 (Training Loop) | 30 min | Cell 3-12 |
| 09:40-10:00 | 第二部分：Tokenizer 分词器 | 20 min | Cell 13-18 |
| 10:00-10:10 | **休息** | 10 min | Cell 19 |
| 10:10-10:30 | 第三部分：Embedding 嵌入 | 20 min | Cell 20-31 |
| 10:30-10:50 | 第四部分：Prompt Engineering | 20 min | Cell 32-39 |
| 10:50-11:05 | 第五部分：企业决策框架 | 15 min | Cell 40-41 |
| 11:05-11:10 | 总结 + 下午预告 | 5 min | Cell 42 |

### 教学节奏提示

- 讲解部分（Markdown Cell）：**慢讲、多互动**，引导学生思考
- 代码演示（Code Cell）：**先讲思路，再运行代码，最后解读输出**
- 练习部分：**给足时间，先让学生独立思考，再逐步给提示**
- 每个部分结束时做 1 分钟小结，确认学生跟上了

### 课前准备清单

- [ ] 确认 Notebook 环境可正常运行（提前跑一遍所有Cell）
- [ ] 确认 API Key 配置正确（Day0 setup 已完成）
- [ ] 准备白板/画板（画训练循环的图）
- [ ] 确认学生已完成 Day0 环境配置

---

## 开场：课程总览与环境检查（09:00-09:10，10分钟）

### Cell 0 - 课程介绍 Markdown

📍 **运行Cell**: Cell 0（Markdown，无需运行）

⏱ **时间**: 5 分钟

🎯 **目的**: 让学生了解三天课程的整体脉络，以及今天上午的学习目标

🗣 **讲解话术**:

> 大家早上好！欢迎来到企业AI大模型培训的第一天。
>
> 我们先看一下这三天要学什么。第一天——也就是今天——我们要搞懂大模型最基础的东西：文本是怎么变成数字的，模型又是怎么训练出来的。
>
> 看一下这个大模型生命周期的图：预训练 → SFT → RLHF → 部署。注意每一步的成本——预训练要数百万美金，SFT几千到几万，RLHF几千，部署是持续成本。**作为企业，我们不需要从头预训练，但必须理解每一步在干什么，才能做出正确的技术决策。**
>
> 今天上午的目标很明确：搞懂五个概念——训练循环、分词器、嵌入、提示工程、以及企业该怎么选方案。到第三天结束，你们会亲手搭一个 RAG + Agent 系统。
>
> 但别急，我们从最基础的开始——神经网络到底是怎么训练的？

👀 **指出要点**:
- 指着三天课程表，强调 Day 1 是基础，Day 2 是实战，Day 3 是落地
- 指着生命周期图，强调成本的数量级差异
- 特别强调："我们今天这个小网络只有641个参数，GPT-4有1.8万亿，但训练原理是一模一样的"

❓ **可能的学生提问**:
- Q: "我们公司没有GPU，能学会吗？" → A: "完全可以！今天所有代码都在CPU上跑，而且企业实际用大模型主要是调API，不需要自己训练。"
- Q: "RAG和微调哪个更适合我们？" → A: "好问题！这正是今天最后一部分要讲的，先别急。"

➡️ **过渡**: "好，先让我们确认一下环境没有问题。"

### Cell 1-2 - 环境配置与依赖加载

📍 **运行Cell**: Cell 1（环境配置）、Cell 2（基础依赖）

⏱ **时间**: 3 分钟

🎯 **目的**: 确认所有学生的环境都能正常工作

🗣 **讲解话术**:

> 我们先跑两个Cell确认环境。Cell 1 是加载配置文件——你们在 Day0 已经配好了 API Key 对吧？
>
> Cell 2 导入基础库：PyTorch 是我们今天用的深度学习框架，numpy 做数学运算，matplotlib 画图。
>
> 大家运行一下，看到 "环境准备完成" 就OK了。**如果报错，举手示意**，我来帮你看。

👀 **检查输出**:
- Cell 1 应显示: `[OK] 已加载配置` + API Key（部分隐藏）+ LLM/Embedding 后端信息
- Cell 2 应显示: PyTorch 版本、CUDA 状态、"环境准备完成"
- **注意**: CUDA available: False 是正常的（CPU环境），不要让学生担心

❓ **常见问题**:
- Q: "CUDA available 是 False，是不是有问题？" → A: "没问题！我们今天的模型很小，CPU完全够用。GPU主要是做大规模训练时才需要。"
- 如果有学生 Cell 1 报错，通常是 Day0 没完成配置，让助教协助

➡️ **过渡**: "环境没问题了，那我们正式开始——来看看神经网络是怎么学习的。"

---

## 第一部分：训练循环 Training Loop（09:10-09:40，30分钟）

### Cell 3 - 训练循环总览 Markdown

📍 **运行Cell**: Cell 3（Markdown，无需运行）

⏱ **时间**: 3 分钟

🎯 **核心概念**: 训练循环的四个步骤 Forward → Loss → Backward → Step

🗣 **讲解话术**:

> 好，这是整个上午最重要的一张图。神经网络的训练，不管是我们今天这个641参数的小网络，还是GPT-4那1.8万亿参数的大模型，核心都是这四步：
>
> **Forward（前向传播）**——把数据喂进去，得到一个预测结果。就像你考试答题。
>
> **Loss（计算损失）**——把预测结果和正确答案比一下，算出差距有多大。就像老师改卷打分。
>
> **Backward（反向传播）**——算出每个参数应该往哪个方向调、调多少。就像你看了试卷分析，知道哪里要改进。
>
> **Step（更新参数）**——真正把参数调一下。就像你根据分析去改进自己的答题方式。
>
> 然后重复。考一次、改一次，再考、再改，反复几百次甚至几百万次，模型就越来越准了。
>
> **这四步就是深度学习的全部核心。** 记住了吗？Forward、Loss、Backward、Step。

👀 **教学技巧**: 建议在白板上画出这个循环图，箭头连成一个环。可以让学生跟着念一遍："前向、损失、反向、更新"

❓ **预判提问**:
- Q: "反向传播是怎么算的？" → A: "用链式法则（微积分里的），PyTorch帮我们自动算了，一行代码 `loss.backward()` 就搞定。我们先理解流程，数学细节不是重点。"

➡️ **过渡**: "说了半天理论，我们来实际看看数据长什么样。"

### Cell 4-5 - 创建月亮形数据集

📍 **运行Cell**: Cell 4（Markdown）、Cell 5（代码 - 生成数据+可视化）

⏱ **时间**: 3 分钟

🎯 **演示目的**: 展示一个无法用直线分开的二分类数据集，说明为什么需要神经网络

🗣 **讲解话术**:

> 我们用 sklearn 的 `make_moons` 生成一个经典的"月亮形"数据集。
>
> 看一下参数：**1000个数据点**，噪声 noise=0.1。生成的数据有两类，蓝色和红色。
>
> 【运行Cell 5，等图出来】
>
> 大家看这个图——两个半月形交错在一起。你能画一条直线把蓝色和红色完全分开吗？不能对吧？**这就叫"线性不可分"**。
>
> 这就是为什么我们需要神经网络——它能学到一条弯弯曲曲的决策边界，把这两类分开。
>
> 注意输出：`X=(1000,2), y=(1000,1)`。X是坐标，两个维度分别是x和y坐标；y是标签，0或1。

👀 **指出输出中的关键信息**:
- 数据形状: `X=[1000, 2]` → 1000个样本，每个样本2个特征（x坐标、y坐标）
- 图中蓝色（类别0）和红色（类别1）明显交错，无法用直线分开
- noise=0.1 让数据有一点点噪声，更接近真实场景

❓ **可能提问**:
- Q: "为什么用这个数据集？" → A: "因为它简单、直观，而且完美展示了神经网络的优势——处理非线性问题。真实世界的数据几乎都是非线性的。"

➡️ **过渡**: "数据准备好了，现在我们来搭建一个小神经网络。"

### Cell 6-7 - 搭建MLP神经网络

📍 **运行Cell**: Cell 6（Markdown - MLP结构图）、Cell 7（代码 - 定义模型）

⏱ **时间**: 4 分钟

🎯 **核心概念**: 多层感知机的结构 + 参数量的概念

🗣 **讲解话术**:

> 我们搭一个三层的 MLP（多层感知机）：输入2维 → 隐藏层1有32个神经元 → 隐藏层2有16个 → 输出1个值。
>
> 中间有 ReLU 激活函数——它做的事很简单：`max(0, x)`，小于0的值变成0，大于0的保持不变。为什么需要它？因为**没有激活函数，多少层线性层堆在一起还是线性的**，就没法处理我们刚才那个月亮形数据了。
>
> 最后一层用 Sigmoid，把输出压到 0~1 之间，表示"属于类别1的概率"。
>
> 【运行Cell 7】
>
> 看输出——模型结构很清楚。最重要的是这一行：**总参数量: 641**。
>
> 641个参数！来，我们做个对比：GPT-4有大约1.8万亿（1,800,000,000,000）个参数。差了多少倍？大约**28亿倍**。但是训练的原理——Forward、Loss、Backward、Step——是完全一样的！
>
> 这就像学开车：你在驾校用桑塔纳练的，上了路开保时捷，方向盘、油门、刹车的原理都一样，只是车大了。

👀 **指出输出关键信息**:
- 模型结构：6层（Linear-ReLU-Linear-ReLU-Linear-Sigmoid）
- **641个参数** — 这个数字要反复强调
- 参数怎么算的：(2*32+32) + (32*16+16) + (16*1+1) = 96+32+528+16+16+1 = 641 （不需要细讲，感兴趣的学生可以自己算）

❓ **预判提问**:
- Q: "ReLU为什么不用Sigmoid？" → A: "ReLU计算更快，而且不会有梯度消失问题。中间层用ReLU，最后一层才用Sigmoid因为我们需要0-1的概率输出。"
- Q: "641个参数是怎么算出来的？" → A: "每个Linear层有 weight矩阵 + bias向量。比如 Linear(2,32) 有 2*32=64 个weight + 32个bias = 96个参数。大家课后可以自己算算看。"

➡️ **过渡**: "模型搭好了，接下来定义怎么衡量模型的好坏——也就是损失函数。"

### Cell 8-9 - 训练循环四步曲 + 损失函数与优化器

📍 **运行Cell**: Cell 8（Markdown - 四步表格）、Cell 9（代码 - 定义criterion和optimizer）

⏱ **时间**: 3 分钟

🎯 **核心概念**: 损失函数（BCELoss）和优化器（Adam）的选择

🗣 **讲解话术**:

> Cell 8 这个表格就是我们之前说的四步，只不过现在有了具体的代码对应。大家先看一眼，等会练习要用到。
>
> 【运行Cell 9】
>
> 这里我们选了两个关键组件：
>
> **BCELoss（二分类交叉熵损失）**——衡量"你预测的概率"和"真实标签"差多远。比如真实标签是1（正类），你预测0.9，loss就很小；你预测0.1，loss就很大。
>
> **Adam优化器**——你可以理解为一个"聪明的参数更新策略"。它会自动调整每个参数的学习速率。在企业中，**Adam是默认的首选优化器**，90%的场景都可以用它。
>
> **学习率0.01**——每次更新参数的步长。太大会震荡，太小会太慢。0.01是个常见的起始值。

👀 **输出要点**:
- BCELoss — 二分类的标准损失
- Adam — 企业中最常用的优化器
- lr=0.01 — 学习率

❓ **预判提问**:
- Q: "学习率怎么选？" → A: "通常从0.001或0.01开始试。学习率太大模型会发散（loss上升），太小训练很慢。Adam的好处是它能自适应调整。"

➡️ **过渡**: "好，所有准备工作做完了。现在轮到你们自己来写训练循环了！"

### Cell 10-11 - 练习1：补全训练循环四步曲

📍 **运行Cell**: Cell 10（Markdown - 练习说明）、Cell 11（代码 - 学生填写）

⏱ **时间**: 10 分钟（含讲解）
- 讲解练习要求: 2 分钟
- 学生独立完成: 5 分钟
- 讲解答案 + 运行: 3 分钟

🎯 **练习目的**: 让学生亲手写出 Forward→Loss→Backward→Step 四行代码，加深印象

🗣 **布置练习的话术**:

> 好，现在轮到你们了。看Cell 10的说明——你们要在Cell 11的代码里补全4行代码。
>
> 就是刚才我们说的四步：
> 1. `outputs = model(X_tensor)` — 前向传播
> 2. `loss = criterion(outputs, y_tensor)` — 算损失
> 3. `optimizer.zero_grad()` 然后 `loss.backward()` — 清空旧梯度，算新梯度
> 4. `optimizer.step()` — 更新参数
>
> 给你们5分钟，先自己试。不会的翻一下Cell 8的表格。实在不行点开提示。

**给提示的节奏**:
- 2分钟后："如果卡住了，看一下Cell 8那个四步表格，代码都在里面。"
- 4分钟后："大部分同学应该差不多了。还没写完的可以点开提示2看关键代码。"
- 5分钟后：统一讲解

🗣 **讲解答案的话术**:

> 好，我们来看答案。其实就是四行（注意 zero_grad 和 backward 是两行）：
> ```python
> outputs = model(X_tensor)          # 前向
> loss = criterion(outputs, y_tensor) # 损失
> optimizer.zero_grad()               # 清空梯度（重要！）
> loss.backward()                     # 反向传播
> optimizer.step()                    # 更新
> ```
>
> 特别注意 `optimizer.zero_grad()` 要在 `loss.backward()` **之前**调用！为什么？因为PyTorch默认会**累加**梯度，不清空的话上一轮的梯度会和这一轮混在一起。

👀 **运行后指出输出关键数据**:
- Epoch 100: Loss约0.0099, Accuracy 1.0000 — 100轮就基本收敛了
- Epoch 500: **Loss约0.0005**, Accuracy 1.0000 — 1000个点全部分对
- 训练耗时约 **0.58秒** — 强调：641个参数用CPU不到1秒
- "验证通过！最终准确率: 1.0000" — 100%准确率

🗣 **运行后的点评**:

> 看到了吗？500轮训练，不到1秒钟，1000个数据点全部分对。Loss从一开始的0.6x降到了0.0005，基本为零了。
>
> 这就是神经网络学习的过程——反复迭代，逐渐变好。GPT-4的训练也是这样，只不过它的循环要跑几百万次，数据是整个互联网，参数有1.8万亿个。

❓ **常见学生错误**:
- 忘了写 `optimizer.zero_grad()` → loss不下降或下降很慢，因为梯度累加了
- 把 `zero_grad()` 放在 `step()` 后面 → 功能上也能工作，但不是最佳实践
- 写成 `model(X)` 而不是 `model(X_tensor)` → 报错，因为numpy数组不能直接用

➡️ **过渡**: "训练过程的数字大家看到了，但数字不够直观。我们来画个图看看。"

### Cell 12 - 可视化训练过程

📍 **运行Cell**: Cell 12（代码 - 双图可视化）

⏱ **时间**: 3 分钟

🎯 **演示目的**: 直观展示 Loss 下降曲线 + 模型学到的决策边界

🗣 **讲解话术**:

> 【运行Cell 12，等两个图出来】
>
> 左边这个图是 **Loss曲线**。横轴是训练轮次（Epoch），纵轴是损失值。看到了吧？一开始Loss很高，然后迅速下降，最后趋于平坦。这就是训练收敛的典型曲线。
>
> 你们以后做训练的时候，**第一件事就是画这个Loss曲线**。如果Loss不下降——要么学习率太小，要么代码有bug。如果Loss震荡——学习率太大。如果Loss先降后升——过拟合了。
>
> 右边这个图更有意思——这是模型学到的**决策边界**。颜色深浅表示模型预测的概率。看到了吗？它学会了一条弯弯曲曲的线，完美把蓝色和红色分开了！
>
> 回忆一下开头说的"线性不可分"——直线做不到的事，神经网络做到了。这就是非线性激活函数（ReLU）的功劳。

👀 **指出图中的关键模式**:
- 左图：Loss在前50-100个Epoch急剧下降，之后逐渐平坦 → 典型的收敛行为
- 左图：浅色线是原始Loss（有波动），深色线是平滑后的 → 实际训练中loss都有波动，很正常
- 右图：决策边界是一条**曲线**，不是直线 → 神经网络的非线性能力
- 右图：红蓝两色区域完美分开 → 100%准确率的视觉验证

❓ **互动问题**:
- "如果我们不加ReLU，决策边界会是什么样？" → 会是一条直线！因为多个线性层叠加还是线性的
- "如果noise调大（比如0.5），会怎样？" → 两类数据会更混在一起，准确率会下降，决策边界更复杂

➡️ **过渡**: "好，训练循环这部分大家掌握了。接下来我们想一个问题——刚才的输入是数字坐标，但大模型处理的是文本啊。文本怎么变成数字呢？这就引出了Tokenizer。"

---

## 第二部分：Tokenizer 分词器（09:40-10:00，20分钟）

### Cell 13 - Tokenizer 总览 Markdown

📍 **运行Cell**: Cell 13（Markdown，无需运行）

⏱ **时间**: 3 分钟

🎯 **核心概念**: 三种分词策略的对比，以及为什么BPE是主流

🗣 **讲解话术**:

> 好，我们进入第二个话题——Tokenizer（分词器）。
>
> 模型只认数字，不认文字。所以第一步就是把文字变成数字。怎么变？先切分成小块（token），每个token对应一个编号（ID）。
>
> 看这个表格，有三种切法：
>
> **字符级**：每个字母一个token。好处是词表很小（英文就26个字母），坏处是序列太长。一句话可能变成上百个token。
>
> **单词级**：每个单词一个token。好处是语义清楚，坏处是词表爆炸——英语有几十万个单词，再加上各种变形、新词，根本管不过来。
>
> **子词级（BPE）**：折中方案！把常见的字符组合合并成子词。比如 "learning" 是一个token，但 "learned" 可能被切成 "learn" + "ed"。这样词表大小可控（通常3-5万），又能处理任何新词。
>
> **GPT、LLaMA、Qwen——所有主流大模型都用BPE。** 这是目前的行业标准。

➡️ **过渡**: "说太多不如看代码。我们来实际体验一下三种分词的差异。"

### Cell 14 - 三种分词策略演示

📍 **运行Cell**: Cell 14（代码 - 英文分词对比）

⏱ **时间**: 3 分钟

🎯 **演示目的**: 用同一句英文展示字符级、单词级、BPE的差异

🗣 **讲解话术**:

> 【运行Cell 14】
>
> 我们用 "Hello, I'm learning machine learning!" 这句话来对比。
>
> 字符级：**37个token**！每个字母、标点、空格都是一个token。序列太长了。
>
> 单词级：**5个token**。但注意——"Hello," 包含了逗号，"learning!" 包含了感叹号。而且 "I'm" 是一个token，以后遇到 "I'll"、"I've" 又是不同的token，词表会无限膨胀。
>
> BPE：**8个token**。看看它怎么切的——"Hello" 是一个token，逗号单独一个，" I" 是一个（注意包含前面的空格），"'m" 是一个。最妙的是 " learning" 出现了两次，token ID都是6975——**同一个子词，同一个ID**。
>
> 这就是BPE的平衡之美：既不像字符级那么碎，也不像单词级那么粗。

👀 **指出输出关键数据**:
- 字符级: **37 tokens** — 太碎了
- 单词级: **5 tokens** — 太粗了，标点粘在单词上
- BPE: **8 tokens** — 恰到好处
- 注意 BPE 中 `learning` 的token ID是6975，出现两次 → 相同子词相同编号
- 注意空格被包含在token里（如 `' I'`, `' learning'`）→ BPE的特点

➡️ **过渡**: "英文看完了，那中文呢？中文没有空格，怎么切分？"

### Cell 15 - 中文分词演示

📍 **运行Cell**: Cell 15（代码 - tiktoken 中文分词）

⏱ **时间**: 2 分钟

🎯 **关键发现**: 中文14个字符 → 14个token，几乎每个字一个token

🗣 **讲解话术**:

> 【运行Cell 15】
>
> 来看中文："你好！我正在学习大语言模型。"——**14个中文字符，生成了14个token**。
>
> 注意看逐个token的解码——大部分中文字是一个字一个token："你"、"好"、"我"、"大"、"语"、"言"、"模"、"型"。
>
> 有一个例外："正在"被合并成一个token（ID 97655）——因为"正在"这个组合在训练数据中出现得足够多，BPE把它合并了。
>
> 还有一个有趣的现象：看"学"后面——"习"被拆成了两个token（18259和254），解码出来是乱码（问号符号）。这是因为cl100k_base是GPT-4的编码器，它在英文数据上训练得多，**对中文的覆盖不够好**。
>
> 这带来一个直接的企业影响——中文处理的token消耗更多，API成本更高！

👀 **指出输出关键数据**:
- 14个中文字符 → 14个token → 平均1.0个token/字符
- "正在" 被合并为1个token — BPE学到了常见组合
- "习" 被拆成2个字节级token（显示为乱码）— cl100k_base对中文支持不佳

➡️ **过渡**: "中文效率这么低，到底有多低？我们来做个量化对比。"

### Cell 16 - 不同语言的Token效率对比

📍 **运行Cell**: Cell 16（代码 - 多语言效率对比）

⏱ **时间**: 3 分钟

🎯 **关键数据**: English 6.22 vs Chinese 0.94 字符/token — 成本差6倍以上！

🗣 **讲解话术**:

> 【运行Cell 16】
>
> 这个对比非常重要，大家仔细看：
>
> **英文**：56个字符只需要9个token，效率 **6.22字符/token**。
> **中文**：15个字符需要16个token，效率 **0.94字符/token**。
>
> 什么意思？同样的语义内容，用中文调API要消耗比英文**多6-7倍的token**！
>
> 代码是3.89，中英混合是1.14。
>
> **企业启示**：如果你的业务主要是中文场景，一定要考虑用中文优化过的模型（比如Qwen、GLM），它们的分词器对中文做了专门优化，一个中文词（两三个字）只需要一个token。这可以直接省下大量API费用。
>
> 说到费用，我们来算算具体能省多少钱。

👀 **指出的关键对比**:
- English: 6.22 字符/token — 最高效
- Chinese: **0.94 字符/token** — 最低效！不到英文的1/6
- Code: 3.89 — 代码介于中间
- Mixed: 1.14 — 混合文本被中文拉低

❓ **互动提问**:
- 可以问学生："如果你们公司每天处理1万条中文客服消息，用cl100k_base和用Qwen的分词器，一个月的API费用差多少？" → 引出练习2

➡️ **过渡**: "我们来做个实际的成本计算，感受一下分词器选择对钱包的影响。"

### Cell 17-18 - 练习2：企业Token成本计算器

📍 **运行Cell**: Cell 17（Markdown - 练习说明）、Cell 18（代码 - 学生填写）

⏱ **时间**: 15 分钟（含讲解）
- 讲解练习要求: 2 分钟
- 学生独立完成: 8 分钟
- 讲解答案 + 运行: 5 分钟

🎯 **练习目的**: 将Token效率差异转化为企业可感知的成本数字

🗣 **布置练习的话术**:

> 这个练习模拟一个真实的企业场景：你们公司每天处理1万次客户查询，每次平均200个中文字符，API价格0.06元/千tokens。
>
> 你们要做两件事：
> 1. 写一个函数，用 tiktoken 的 cl100k_base 编码器计算实际token数，算出日/月成本
> 2. 写一个函数，模拟中文优化的分词器（假设每个字符只需0.6个token），算出日/月成本
> 3. 对比两者，看看每月能省多少钱
>
> 核心代码就是 `enc.encode(text)` 获取token列表，`len()` 算数量，然后做简单的乘除法。
>
> 给你们8分钟。

**给提示的节奏**:
- 3分钟后："关键思路：先算平均每条查询的token数，乘以日查询量，再乘以单价。"
- 5分钟后："如果卡住了，点开提示2看关键代码。"
- 8分钟后：统一讲解

🗣 **讲解答案的话术**:

> 好，我们来看结果：
>
> cl100k_base：日token量 **338,000**，月成本 **¥608.40**
> 中文优化：日token量 **196,800**，月成本 **¥354.24**
>
> 差多少？**每月省¥254.16**！一年就是3000多块。而且这只是输入token的成本，还没算输出。
>
> 而且我们模拟的场景是每天1万次查询，如果是百万级的呢？那就是每月省几万块。
>
> 所以**选择合适的分词器/模型，是企业AI成本优化的第一步**。这不是技术问题，是钱的问题！

👀 **关键数字**:
- cl100k_base 月成本: ¥608.40
- 中文优化 月成本: ¥354.24
- **月节省: ¥254.16** — 这个数字要强调

❓ **可能提问**:
- Q: "0.6个token/字符这个假设准确吗？" → A: "这是一个合理的估计。Qwen的分词器在中文上大约是0.5-0.7个token/字符，具体取决于文本内容。实际使用时应该用真实数据测试。"
- Q: "我们公司应该用哪个模型？" → A: "如果主要是中文场景，建议优先考虑中文优化的模型，比如Qwen、GLM。不仅token效率高，中文理解能力也更好。"

➡️ **过渡**: "好，关于Token和成本大家有了直观认识。现在我们休息10分钟，回来讲一个更深层的问题——Token ID只是编号，模型怎么理解语义？"

---

## 休息提醒（10:00-10:10）

📍 **对应Cell**: Cell 19

🗣 **话术**:

> 休息10分钟！大家上个厕所、倒杯水。
>
> 休息的时候可以想一个问题：Token ID只是一个数字编号，比如"你"是57668，"好"是53901。这些数字之间有什么数学关系吗？57668和53901这两个数字能告诉模型"你"和"好"有什么语义关系吗？
>
> 答案是：不能。回来我们讲怎么解决这个问题。

**休息期间**：
- 检查有没有学生环境出问题，趁休息时间帮忙解决
- 如果有学生提前做完了练习2，可以建议他们试试改参数（比如改日查询量、改单价）看看结果变化

---

## 第三部分：Embedding 嵌入（10:10-10:30，20分钟）

### Cell 20 - Embedding 总览 Markdown

📍 **运行Cell**: Cell 20（Markdown，无需运行）

⏱ **时间**: 2 分钟

🎯 **核心概念**: Embedding = 给每个Token分配一个向量，让语义相似的词向量也相似

🗣 **讲解话术**:

> 好，回来了。刚才我让大家想的问题——Token ID只是编号，编号本身没有语义。57668和53901之间的数学关系不能告诉模型"你"和"好"有什么关系。
>
> 怎么办？我们给每个token分配一个**向量**（一组数字），叫做 **Embedding**。
>
> 比如"猫"的向量是 [0.2, -0.1, 0.8, ...]，"狗"的向量是 [0.3, -0.2, 0.7, ...]。这两个向量很接近——因为猫和狗语义上是相关的（都是动物、宠物）。而"桌子"的向量可能是 [−0.5, 0.9, −0.3, ...]，和猫狗就差很远。
>
> `nn.Embedding` 是什么？说白了就是一个**查表矩阵**。形状是 [词表大小, 向量维度]。给一个Token ID，就返回对应行的向量。就这么简单！
>
> **没有复杂计算，就是查表。** 我们马上来验证这一点。

➡️ **过渡**: "来看代码。"

### Cell 21-22 - Embedding就是查表

📍 **运行Cell**: Cell 21（代码 - 创建Embedding矩阵）、Cell 22（代码 - 证明查表等价性）

⏱ **时间**: 4 分钟

🎯 **核心证明**: embedding(idx) 和 weight[idx] 完全等价 → Embedding就是矩阵索引

🗣 **讲解话术**:

> 【运行Cell 21】
>
> 看，我们创建了一个 Embedding，词表大小10，每个词用4维向量表示。所以Embedding矩阵是 10x4 的。
>
> 矩阵里的数字都是随机初始化的——这些数字后面会通过训练来学习。
>
> 【运行Cell 22】
>
> **这个Cell很关键！** 我们用两种方式查词向量：
> - 方法1：`embedding(word_idx)` — 用Embedding层的forward方法
> - 方法2：`embedding.weight[word_idx]` — 直接对矩阵做索引
>
> 结果**完全一样**！`torch.allclose` 返回 True。
>
> 这说明什么？**nn.Embedding本质就是矩阵的行索引操作**，没有任何神经网络的前向计算、没有激活函数、没有什么花里胡哨的东西。就是一个大表，给编号，查出对应的一行。
>
> 但它的威力在于——这个表的值是通过训练学到的！训练之后，语义相似的词，向量就会很接近。

👀 **指出输出关键信息**:
- Cell 21: Embedding矩阵形状 `[10, 4]` — 10个词，每个4维
- Cell 22: 两种方法输出的张量**数值完全相同**
- `两种方法完全等价: True` — 这一行要重点强调

❓ **互动提问**:
- "大家猜一下，GPT-4的Embedding矩阵有多大？" → 词表约10万 x 维度约12288 = 约12亿个参数！光Embedding矩阵就比我们刚才整个MLP大200万倍。

➡️ **过渡**: "Embedding矩阵里的值是随机初始化的，怎么训练出有意义的值呢？答案是通过预测下一个词。"

### Cell 23-27 - 训练词向量（Bigram模型）

📍 **运行Cell**: Cell 23（Markdown - 训练原理）、Cell 24（代码 - 准备数据）、Cell 25（代码 - Bigram模型）、Cell 26（代码 - 训练）、Cell 27（代码 - PCA可视化）

⏱ **时间**: 7 分钟

🎯 **核心概念**: 通过"预测下一个词"的任务来训练Embedding → 相似上下文的词获得相似向量

🗣 **讲解话术**:

> 【先讲Cell 23的原理】
>
> 训练Embedding的核心思想很简单：**用当前词预测下一个词**。
>
> 如果"机器"后面经常跟"学习"，"深度"后面也经常跟"学习"，那训练之后"机器"和"深度"的向量就会很接近。为什么？因为模型要做出相似的预测，就需要让输入的向量也相似。
>
> 这就是Word2Vec/语言模型训练词向量的核心思想。GPT做的事情本质也是这个——预测下一个token。
>
> 【运行Cell 24】
>
> 我们用5句中文来训练。按字符分词后，得到28个不同的字（词表大小28），51个训练样本（相邻字对）。
>
> 看前5个样本："机"→"器"，"器"→"学"，"学"→"习"... 就是相邻两个字的配对。
>
> 【运行Cell 25】
>
> Bigram模型很简单：一个Embedding层（28x16）+ 一个线性层（16→28）。总参数量 **924个**。
>
> 工作原理：输入一个字的ID → 查Embedding得到16维向量 → 线性层输出28个概率 → 预测下一个字是哪个。
>
> 【运行Cell 26】
>
> 训练500轮，看Loss从大约3.3降到了 **0.288**。这个Loss不会降到0，因为中文里"是"后面可以跟"人"也可以跟"机"也可以跟"深"，一个字有多个可能的下一个字，不可能100%预测对。
>
> 【运行Cell 27】
>
> 这个图就是训练后的词向量空间！用PCA把16维降到2维来可视化。
>
> 大家看看——哪些字聚在一起？"机"和"器"靠得近对吧？因为它们总是一起出现（"机器"）。类似的，"学"和"习"也应该比较近。
>
> 当然我们的训练数据只有5句话，51个样本，效果不会很惊艳。但想象一下用整个维基百科来训练——几十亿个样本——训练出来的词向量就能精准捕捉各种语义关系了。

👀 **关键数据点**:
- Cell 24: 词表大小 **28**，训练样本 **51个** — 很小的数据集
- Cell 25: Bigram模型参数量 **924** — 比MLP还大一点，因为词表是28不是2
- Cell 26: 最终Loss **0.2881** — 不会降到0是正常的
- Cell 27: PCA图中观察语义聚类 — 常一起出现的字靠近

❓ **可能提问**:
- Q: "这和Word2Vec有什么关系？" → A: "本质思想一样——通过上下文来学习词的表示。Word2Vec用的是CBOW或Skip-gram，我们这里用的是Bigram（只看前一个词），GPT用的是完整的Transformer。但核心思想不变：出现在相似上下文中的词应该有相似的向量。"
- Q: "为什么Loss不降到0？" → A: "因为一个字后面可能跟不同的字。比如'是'后面可以是'人'也可以是'机'也可以是'深'，模型不可能100%确定下一个字是什么，所以Loss不会为0。这叫做数据中的固有不确定性（entropy）。"

➡️ **过渡**: "我们用5句话训练出了简单的词向量。但实际怎么衡量两个向量相不相似呢？这就需要一个数学工具——余弦相似度。"

### Cell 28-29 - 练习3：实现余弦相似度

📍 **运行Cell**: Cell 28（Markdown - 练习说明）、Cell 29（代码 - 学生填写）

⏱ **时间**: 10 分钟（含讲解）
- 讲解公式和要求: 2 分钟
- 学生独立完成: 5 分钟
- 讲解答案 + 运行: 3 分钟

🎯 **练习目的**: 理解并实现余弦相似度——这是向量检索（RAG）的基础

🗣 **布置练习的话术**:

> 余弦相似度是衡量两个向量相似程度的标准工具。公式很简单：
>
> **cos_sim(a,b) = a点乘b / (|a| * |b|)**
>
> 结果范围是 -1 到 1：1 表示方向完全相同，0 表示正交（无关），-1 表示方向完全相反。
>
> 你们要做的就是用 `np.dot()` 算点乘，`np.linalg.norm()` 算向量长度，然后除一下。就3行代码。
>
> 做完基础验证后，用这个函数去查刚才训练出来的词向量，看看跟"学"和"机"最相似的字是哪些。
>
> 5分钟时间，开始！

**给提示的节奏**:
- 2分钟后："核心就3行：dot=np.dot(v1,v2)，norm1=np.linalg.norm(v1)，return dot/(norm1*norm2)"
- 5分钟后：统一讲解

🗣 **讲解答案的话术**:

> 答案非常简洁：
> ```python
> dot_product = np.dot(v1, v2)
> norm1 = np.linalg.norm(v1)
> norm2 = np.linalg.norm(v2)
> return dot_product / (norm1 * norm2)
> ```
>
> 验证通过了：相同向量的余弦相似度是1.0，相反向量是-1.0。
>
> 看一下和"学"最相似的字——结果可能不太完美，因为我们的训练数据太少了（只有5句话）。但方法是对的。
>
> **余弦相似度这个工具你们后面会反复用到**，特别是在RAG（检索增强生成）里——用它来找跟用户查询最相似的文档片段。Day 3 我们会详细讲。

👀 **输出关键信息**:
- `cos_sim(v, v) = 1.0000` — 正确
- `cos_sim(v, -v) = -1.0000` — 正确
- 相似词的结果受限于小数据集，语义关系不是特别明显，但方法正确

❓ **常见学生错误**:
- 忘了除以范数 → 结果不在 [-1, 1] 范围内
- 用 `*` 代替 `np.dot()` → 对数组做的是逐元素乘法，不是点乘

➡️ **过渡**: "刚才我们用5句话训练的词向量效果一般。实际企业中怎么办？用预训练好的Embedding模型！"

### Cell 30-31 - 练习4：用预训练Embedding找语义相似句

📍 **运行Cell**: Cell 30（Markdown - 练习说明）、Cell 31（代码 - 学生填写）

⏱ **时间**: 10 分钟（含讲解）
- 讲解要求 + 讲解预训练Embedding概念: 2 分钟
- 学生独立完成: 5 分钟
- 讲解答案 + 运行: 3 分钟

🎯 **练习目的**: 体验真实的预训练Embedding模型（text-embedding-v3, 1024维），感受语义检索的效果

🗣 **布置练习的话术**:

> 这个练习是连接刚才学的知识和真实企业应用的桥梁。
>
> 我们要用阿里百炼的 `text-embedding-v3` 模型——一个预训练好的Embedding模型，输出**1024维**的向量。比我们刚才的4维、16维高了不知道多少。
>
> 任务很简单：给5句话和一个查询"AI技术帮助企业提高效率"，算出查询和每句话的余弦相似度，按高到低排序。
>
> 代码就几步：`embedder.embed(sentences)` 获取向量，然后用你刚才写的 `cosine_similarity` 函数算相似度。

🗣 **讲解答案的话术**:

> 【运行Cell 31】
>
> 看结果！查询是"AI技术帮助企业提高效率"：
>
> 最相似的是"数据挖掘帮助企业做决策"（**0.62**）——因为都涉及"帮助企业"和技术应用。
> 第二是"机器学习可以分析大量数据"（**0.57**）——也是AI技术。
> 第三是"深度学习是人工智能的重要分支"（**0.52**）——AI相关但更偏学术。
>
> 最不相关的是"公园里的花开了"（**0.30**）和"今天的天气非常好"（**0.39**）——完全不相关的日常话题。
>
> 这个结果非常合理对吧？模型完全理解了语义，不是在做关键词匹配！"数据挖掘"和"AI技术"没有共同的关键词，但模型知道它们语义相关。
>
> **这就是RAG的基础！** 用户问一个问题，我们用Embedding算相似度，找出最相关的文档片段，喂给大模型生成回答。Day 3 会详细实战。

👀 **关键数据**:
- text-embedding-v3, 维度 **1024**
- 最高相似度: "数据挖掘帮助企业做决策" → **0.6218** — 语义最接近
- 最低相似度: "公园里的花开了" → **0.3033** — 语义最远
- 排序完全符合人类直觉 — 证明预训练Embedding的强大

❓ **关键教学点**:
- 强调：这不是关键词匹配！"数据挖掘"和"AI技术"没有共同字词，但语义上最相关
- 强调：这就是RAG系统的检索部分

➡️ **过渡**: "好，我们搞懂了文本怎么变成向量。但是有了大模型之后，很多任务其实不需要训练模型——只需要写好Prompt就行了。"

---

## 第四部分：Prompt Engineering 提示工程（10:30-10:50，20分钟）

### Cell 32-33 - Prompt Engineering 总览 + LLM初始化

📍 **运行Cell**: Cell 32（Markdown - 三种策略表格）、Cell 33（代码 - 初始化LLM）

⏱ **时间**: 2 分钟

🎯 **核心概念**: 三种Prompt策略 — Zero-shot、Few-shot、CoT

🗣 **讲解话术**:

> 接下来是最实用的一部分——**Prompt Engineering**。
>
> 有了大模型之后，很多NLP任务不需要自己训练模型了，写好提示词就能搞定。这对企业来说是巨大的好消息——**不需要ML工程师，产品经理都能做AI应用**。
>
> 三种基本策略：
> - **Zero-shot**：直接告诉模型要做什么，不给示例。最简单。
> - **Few-shot**：给几个输入-输出的示例，让模型学着做。更稳定。
> - **CoT（Chain-of-Thought）**：让模型一步步推理。适合复杂问题。
>
> 【运行Cell 33 初始化LLM】
>
> 看到"LLM 就绪"就说明API连接正常。我们来实际感受一下三种策略的效果。

➡️ **过渡**: "来看第一种——Zero-shot。"

### Cell 34 - Zero-shot 演示

📍 **运行Cell**: Cell 34（代码 - Zero-shot情感分类）

⏱ **时间**: 2 分钟

🎯 **演示**: 不给任何示例，直接让模型做情感分类

🗣 **讲解话术**:

> 【运行Cell 34】
>
> 最简单的方式——直接告诉模型"请分类为正面/负面/中性"，然后给文本。
>
> 输入："这个产品质量太差了，用了一周就坏了"
> 输出：**"负面"**
>
> 完美！一个字——负面。模型理解了任务，也给出了正确答案。
>
> 但 Zero-shot 有个问题：输出格式不一定稳定。有时候模型会回复"这是一条负面评价，因为..."，多了一堆解释。在生产环境中，不稳定的输出会让下游解析出错。

➡️ **过渡**: "怎么让输出更稳定？给示例！"

### Cell 35 - Few-shot 演示

📍 **运行Cell**: Cell 35（代码 - Few-shot情感分类）

⏱ **时间**: 3 分钟

🎯 **演示**: 给3个示例后再提问，输出格式更稳定

🗣 **讲解话术**:

> 【运行Cell 35】
>
> Few-shot：我们给了3个示例——正面、负面、中性各一个——然后再问一个新的。
>
> 注意看Prompt的结构：
> 1. 任务描述
> 2. 三个"反馈→分类"的示例
> 3. 最后一个待分类的反馈，后面跟"分类:"让模型补全
>
> 输出：**"正面"**。正确！而且注意输出格式——就一个词，和我们示例中的格式完全一致。
>
> **这就是Few-shot的威力**：通过示例，你不仅教会了模型任务是什么，还教会了它输出的格式。在企业中，**Few-shot是最常用的策略**，因为它在准确率和稳定性之间取得了很好的平衡。

👀 **强调的要点**:
- 注意Prompt的结构设计——任务描述 + 示例 + 待处理输入
- 示例要覆盖不同类别（正面、负面、中性各一个）
- 输出格式跟示例保持一致

➡️ **过渡**: "如果遇到更复杂的判断呢？比如一段话里既有正面又有负面？这时候需要CoT。"

### Cell 36 - Chain-of-Thought 演示

📍 **运行Cell**: Cell 36（代码 - CoT情感分类）

⏱ **时间**: 3 分钟

🎯 **演示**: 让模型分步推理，适合复杂判断场景

🗣 **讲解话术**:

> 【运行Cell 36】
>
> 这个反馈比较复杂："产品功能很强大，但是价格有点贵，总体来说还是值得购买的"。既有正面又有负面，怎么判断？
>
> 我们在Prompt里要求模型分四步分析：
> 1. 找正面表述
> 2. 找负面表述
> 3. 判断整体倾向
> 4. 给出最终分类
>
> 看模型的回复——非常有条理！它找出了正面（"功能很强大"、"值得购买"）和负面（"价格有点贵"），然后分析句子结构（"正面+转折+负面+总结性正面"），最终判断为**正面**。
>
> **CoT的精髓是什么？是把复杂问题拆解成小步骤，让模型一步步思考**。这和人类的思维方式一样——遇到复杂问题，先分析再下结论，而不是直接给答案。
>
> 在企业中，CoT特别适合需要解释推理过程的场景，比如风控决策、医疗诊断建议——不仅要给结果，还要说明为什么。

👀 **模型回复中值得关注的**:
- 模型准确识别了正面和负面表述
- 分析了句子结构（"正面+转折+负面+总结性正面"）
- 最终分类为**正面** — 合理！总结句是正面的

➡️ **过渡**: "三种策略都看完了，来看Cell 37的实践要点表格。"

### Cell 37 - Prompt工程企业实践要点

📍 **运行Cell**: Cell 37（Markdown - 实践要点表格）

⏱ **时间**: 2 分钟

🗣 **讲解话术**:

> 这5个技巧是企业实践中最常用的：
>
> 1. **明确角色**——"你是一个专业的客服分类系统"。给模型一个人设，它的输出会更专业、更一致。
>
> 2. **限制输出格式**——"只回复：正面/负面/中性"。不限制的话模型可能会输出一大段解释，在生产环境中不好解析。
>
> 3. **给示例**——Few-shot永远比Zero-shot稳定。哪怕只给2-3个示例，效果就好很多。
>
> 4. **分步推理**——复杂任务用CoT。但简单任务别用CoT，反而会变慢、费token。
>
> 5. **控制temperature**——分类、提取等确定性任务用 **temperature=0**（不要随机性）；创意写作、头脑风暴用 **0.7+**（要多样性）。
>
> 特别强调第5点——很多初学者不知道temperature。这一个参数调对了，效果能差很多。

➡️ **过渡**: "说了这么多，到底Few-shot和Zero-shot差多少？我们来做个A/B测试量化一下。"

### Cell 38-39 - 练习5：Prompt工程A/B测试

📍 **运行Cell**: Cell 38（Markdown - 练习说明）、Cell 39（代码 - 学生填写）

⏱ **时间**: 15 分钟（含讲解）
- 讲解练习要求: 2 分钟
- 学生独立完成: 8 分钟（这个练习代码量稍多）
- 讲解答案 + 运行: 5 分钟

🎯 **练习目的**: 体验A/B测试的方法论 + 感受Few-shot vs Zero-shot的实际差异

🗣 **布置练习的话术**:

> 这个练习模拟企业中常见的A/B测试场景。你们要：
> 1. 设计一个Few-shot的Prompt模板（带2-3个示例）
> 2. 设计一个Zero-shot的Prompt模板（只有指令）
> 3. 用8条测试数据分别跑两个Prompt，比较准确率
>
> 分类类别是6个：产品质量/物流配送/售后服务/价格争议/账户问题/其他
>
> 关键代码就是 `llm.generate(prompt.format(text=text), temperature=0)`
>
> 然后用 `expected in result` 判断是否正确（因为模型可能回复"产品质量"或"分类：产品质量"，只要包含就算对）。
>
> 给你们8分钟，注意这个练习会调16次API（8条数据 x 2个Prompt），可能需要等一会。

**给提示的节奏**:
- 3分钟后："Prompt模板用f-string的 `{text}` 占位符，这样后面可以 `.format(text=投诉内容)`"
- 5分钟后："核心循环就是 for text, expected in complaints，调两次generate，比较结果"
- 8分钟后：统一讲解

🗣 **讲解答案的话术**:

> 【运行Cell 39，等待约30秒】
>
> 看结果——8条测试数据：
>
> Prompt A（Few-shot）：**8/8 (100%)**
> Prompt B（Zero-shot）：**8/8 (100%)**
>
> 都是100%！你们可能会想："那Few-shot不是没用吗？" 
>
> 别急着下结论。原因有两个：
> 1. **这8条测试数据太简单了**，分类类别很明确，模型做起来没难度。真实场景中会有很多歧义样本，那时候Few-shot的优势就出来了。
> 2. **即使准确率一样，Few-shot的输出格式更稳定**。Zero-shot有时候会输出"这属于产品质量方面的问题"，Few-shot基本都只回复类别名。
>
> 在企业实际落地中，**Always start with Few-shot**。成本不高（多几个示例的token费），但稳定性好很多。

👀 **关键结果**:
- 两种策略在简单测试集上都达到 **100%准确率**
- 实际企业场景中，Few-shot通常在边界case上表现更好
- 强调A/B测试的方法论——不要凭感觉选策略，要用数据说话

❓ **可能提问**:
- Q: "既然都100%，为什么要用Few-shot？" → A: "因为真实数据更复杂、更有歧义。而且Few-shot的输出格式更一致，对下游处理更友好。少量示例不会增加多少成本。"
- Q: "temperature=0是什么意思？" → A: "让模型输出最确定的答案，不引入随机性。分类任务必须用0。"
- Q: "这个A/B测试在企业里怎么做？" → A: "标注一批测试集（几百条），分别用不同的Prompt跑，比较准确率、延迟、成本。上线后还要持续监控。"

➡️ **过渡**: "Prompt Engineering能搞定很多任务，但不是所有任务。什么时候该用Prompt，什么时候该用RAG，什么时候该微调？我们来建立一个决策框架。"

---

## 第五部分：企业决策框架（10:50-11:05，15分钟）

### Cell 40 - 决策框架 Markdown

📍 **运行Cell**: Cell 40（Markdown - 对比表格 + 决策流程图）

⏱ **时间**: 8 分钟

🎯 **核心内容**: 三种方案（Prompt/RAG/微调）的全面对比 + 决策流程图

🗣 **讲解话术**:

> 这部分是今天上午**对企业最直接有用的内容**。
>
> 看这个对比表格——三种方案在成本、周期、数据需求、准确率、可定制性、维护成本上的差异。
>
> **Prompt Engineering**：成本最低（只有API费），开发最快（小时级），不需要训练数据。但准确率中等，定制化能力有限。**80%的企业场景应该从这里开始。**
>
> **RAG（检索增强生成）**：中等成本（向量数据库+API），天级开发周期。最大优势是可以用企业的专有数据，而且知识库更新不需要重新训练。Day 3 我们会详细做。
>
> **微调（Fine-tuning）**：成本最高（GPU+标注数据），周到月级。但准确率最高，定制性最强。Day 2 我们会讲SFT和LoRA。
>
> 再看决策流程图——这是我建议你们贴在办公桌上的：
>
> **第一步永远是试Prompt**。能搞定就不要做更复杂的事。
>
> **需要专有知识？** 比如公司的产品手册、内部政策——用RAG。文档少于100页的甚至可以直接塞进context（现在模型支持100K+的上下文长度），不需要搭向量数据库。
>
> **需要特定风格或格式？** 先用Few-shot Prompt试试。不够好再考虑微调。
>
> **需要极高准确率+有大量标注数据？** 这时候才上微调。
>
> 记住那个**快速决策清单**，5条规则：
> 1. 先试Prompt（80%够用）
> 2. 数据少于100条 → Few-shot > Fine-tuning
> 3. 需要实时知识 → RAG是唯一选择
> 4. 对延迟敏感 → 微调小模型 > 调大模型API
> 5. 预算有限 → Prompt > RAG > 微调

👀 **反复强调的观点**:
- "先试Prompt"——不要一上来就想着微调
- 成本梯度：Prompt（最便宜）→ RAG（中等）→ 微调（最贵）
- 80%的企业AI任务用好Prompt就够了

❓ **互动环节**:
- 可以出几个场景让学生判断该用哪个方案，预热Cell 41的内容
- "你们公司的客服系统应该用哪种方案？" → 引导学生思考

➡️ **过渡**: "我们来看几个具体的企业场景。"

### Cell 41 - 企业场景决策练习

📍 **运行Cell**: Cell 41（代码 - 4个场景示例）

⏱ **时间**: 5 分钟

🎯 **教学方式**: 先让学生猜，再揭晓答案

🗣 **讲解话术**:

> 我来出4个场景，你们先想想该用哪个方案，然后我们一起看答案。
>
> **场景1**：电商客服自动回复，需要查订单信息和退换货政策。
> → 【停顿3秒让学生思考】
> → 答案：**RAG**！因为需要实时查订单数据+公司政策文档，这些信息Prompt里放不下，也不是模型训练数据里有的。
>
> **场景2**：把客户邮件翻译成英文。
> → 答案：**Prompt Engineering**！翻译是大模型的通用能力，不需要任何额外处理。
>
> **场景3**：医疗影像报告自动生成，要符合特定医院格式。
> → 答案：**微调**！医疗场景需要极高准确率，特定格式，专业术语，而且通常有大量历史报告可以用来训练。
>
> **场景4**：内部知识库问答，500页文档。
> → 答案：**RAG**！500页放不进context（即使是128K的模型也不一定够），需要向量数据库做检索。
>
> 【运行Cell 41 确认答案】

👀 **教学技巧**:
- 每个场景先停顿让学生思考，鼓励举手回答
- 如果学生回答错了，不要直接否定，问"为什么这样想？"然后引导
- 强调：没有绝对正确的答案，有时候组合使用（如RAG+微调）更好

❓ **延伸讨论**:
- Q: "能不能RAG+微调结合？" → A: "可以！很多企业先用RAG解决知识问题，再微调模型提升特定领域的表现。但要注意成本和复杂度。"
- Q: "我们公司的XX场景该怎么做？" → A: 如果时间允许，可以讨论1-2个学生的实际场景

➡️ **过渡**: "好，我们来做今天上午的总结。"

---

## 总结与下午预告（11:05-11:10，5分钟）

### Cell 42 - 上午总结

📍 **运行Cell**: Cell 42（Markdown - 总结表格）

⏱ **时间**: 5 分钟

🎯 **总结全部内容 + 预告下午**

🗣 **总结话术**:

> 好，我们来回顾一下今天上午走过的路：
>
> ```
> 文本 → Tokenizer → Token IDs → Embedding → 向量 → 模型 → 输出
> ```
>
> 这条链路就是大模型处理文本的完整路径。你们现在已经理解了每一步在做什么。
>
> 5个关键收获：
>
> 1. **训练循环**——Forward → Loss → Backward → Step。641个参数和1.8万亿参数，原理一模一样。
>
> 2. **Tokenizer**——BPE把文本切成子词。中文效率低（0.94字符/token），选中文优化模型可以省钱（每月省¥254）。
>
> 3. **Embedding**——就是查表矩阵。通过预测下一个词来训练，让语义相似的词有相似的向量。这是RAG的基础。
>
> 4. **Prompt Engineering**——80%的任务用好Prompt就够了。Zero-shot最简单，Few-shot最实用，CoT适合复杂推理。temperature=0用于确定性任务。
>
> 5. **企业决策**——先Prompt → 不够就RAG → 最后考虑微调。这是省钱、省时的正确路径。
>
> **下午预告**：我们要深入Transformer架构，理解让GPT/Qwen/LLaMA如此强大的核心机制——**Self-Attention（自注意力）**。简单说，Self-Attention让模型在处理每个词的时候，能"看到"整个句子里所有其他词，从而理解上下文。这是2017年"Attention is All You Need"论文提出的革命性想法。
>
> 大家吃个午饭，下午见！

👀 **最后强调**:
- 确认学生能说出训练循环的四步
- 确认学生理解 Token → Embedding → 向量 的转换链路
- 确认学生记住"先试Prompt"的决策原则

---

## 附录：Cell编号速查表

| Cell编号 | 类型 | 内容 | 预计时间 |
|----------|------|------|----------|
| 0 | Markdown | 课程总览、学习目标、大模型生命周期 | 5 min |
| 1 | Code | 环境配置（加载.env） | 1 min |
| 2 | Code | 基础依赖导入 | 1 min |
| 3 | Markdown | 训练循环总览（四步图） | 3 min |
| 4 | Markdown | 1.1 创建数据集说明 | 0.5 min |
| 5 | Code | 生成月亮形数据集 + 可视化 | 2.5 min |
| 6 | Markdown | 1.2 MLP结构图 | 1 min |
| 7 | Code | 定义神经网络（641参数） | 3 min |
| 8 | Markdown | 1.3 训练循环四步表格 | 1 min |
| 9 | Code | 定义BCELoss + Adam优化器 | 2 min |
| 10 | Markdown | 练习1说明（训练循环） | 2 min |
| 11 | Code | 练习1代码（学生填写） | 8 min |
| 12 | Code | 可视化：Loss曲线 + 决策边界 | 3 min |
| 13 | Markdown | Tokenizer总览（三种策略表格） | 3 min |
| 14 | Code | 三种分词策略对比（37/5/8 tokens） | 3 min |
| 15 | Code | 中文分词演示（14字符→14 tokens） | 2 min |
| 16 | Code | 多语言Token效率对比（6.22 vs 0.94） | 3 min |
| 17 | Markdown | 练习2说明（Token成本计算器） | 2 min |
| 18 | Code | 练习2代码（¥608 vs ¥354，省¥254/月） | 13 min |
| 19 | Markdown | 休息10分钟 | 10 min |
| 20 | Markdown | Embedding总览（查表+语义空间） | 2 min |
| 21 | Code | Embedding矩阵创建（10x4） | 2 min |
| 22 | Code | 证明Embedding=查表（两方法等价） | 2 min |
| 23 | Markdown | 训练词向量原理（预测下一个词） | 2 min |
| 24 | Code | 准备训练数据（28词表，51样本） | 1 min |
| 25 | Code | Bigram模型定义（924参数） | 1 min |
| 26 | Code | 训练Bigram（500轮，loss→0.288） | 1 min |
| 27 | Code | PCA可视化词向量空间 | 2 min |
| 28 | Markdown | 练习3说明（余弦相似度） | 2 min |
| 29 | Code | 练习3代码（学生填写） | 8 min |
| 30 | Markdown | 练习4说明（预训练Embedding） | 2 min |
| 31 | Code | 练习4代码（text-embedding-v3, 1024维） | 8 min |
| 32 | Markdown | Prompt Engineering总览（三策略表格） | 1 min |
| 33 | Code | 初始化LLM后端 | 1 min |
| 34 | Code | Zero-shot演示 → 回复"负面" | 2 min |
| 35 | Code | Few-shot演示 → 回复"正面" | 3 min |
| 36 | Code | CoT演示 → 分步分析，回复"正面" | 3 min |
| 37 | Markdown | Prompt工程企业实践要点（5技巧） | 2 min |
| 38 | Markdown | 练习5说明（A/B测试） | 2 min |
| 39 | Code | 练习5代码（8/8 100%两种都OK） | 13 min |
| 40 | Markdown | 企业决策框架（对比表+流程图+清单） | 8 min |
| 41 | Code | 4个企业场景决策示例 | 5 min |
| 42 | Markdown | 上午总结 + 下午预告 | 5 min |

---

## 附录：关键数字速记卡

讲课时需要随口说出的数字，提前记住：

| 数字 | 含义 | 出处 |
|------|------|------|
| 1000 | make_moons数据点数 | Cell 5 |
| 641 | MLP模型参数量 | Cell 7 |
| 1.8万亿 | GPT-4参数量（对比用） | Cell 7 |
| 0.0005 | 训练500轮后的Loss | Cell 11 |
| 100% | 训练后准确率 | Cell 11 |
| 37/5/8 | 字符/单词/BPE的token数 | Cell 14 |
| 14→14 | 中文字符数→token数 | Cell 15 |
| 6.22 vs 0.94 | 英文vs中文 字符/token效率 | Cell 16 |
| ¥608 vs ¥354 | cl100k vs 中文优化 月成本 | Cell 18 |
| ¥254 | 月节省金额 | Cell 18 |
| 28 | Bigram模型词表大小 | Cell 24 |
| 924 | Bigram模型参数量 | Cell 25 |
| 0.288 | Bigram训练后Loss | Cell 26 |
| 1024 | text-embedding-v3向量维度 | Cell 31 |
| 0.62 | "数据挖掘帮助企业做决策"的相似度（最高） | Cell 31 |
| 0.30 | "公园里的花开了"的相似度（最低） | Cell 31 |
| 8/8 100% | A/B测试两种Prompt的准确率 | Cell 39 |